# Apache Airflow TaskFlow API: @task Decorator

## Overview

The `@task` decorator transforms Python functions into Airflow tasks. It eliminates the need for traditional operators like `PythonOperator`, providing a cleaner, more intuitive way to define task logic.

### Key Benefits
- No operator boilerplate required
- Automatic XCom serialization/deserialization
- Direct Python function calls
- Type hints and IDE support

## 1. Basic @task Implementation

In [ ]:
from airflow.decorators import dag, task
from datetime import datetime

@dag(
    dag_id='basic_task_example',
    start_date=datetime(2026, 9, 1),
    schedule_interval='@daily',
    catchup=False
)
def basic_task_dag():
    """Simple example of @task decorator."""
    
    @task
    def extract_data():
        """Extract data from source."""
        return {'records': 100, 'source': 'database'}
    
    @task
    def process_data(data):
        """Process extracted data."""
        count = data['records']
        print(f'Processing {count} records')
        return {'processed': count}
    
    # Define dependencies through function calls
    data = extract_data()
    process_data(data)

dag_instance = basic_task_dag()

## 2. Task Configuration Options

In [ ]:
from airflow.decorators import dag, task
from datetime import datetime, timedelta

@dag(
    dag_id='configured_task_example',
    start_date=datetime(2026, 9, 1),
    schedule_interval='@daily',
    catchup=False
)
def configured_task_dag():
    """Demonstrates @task configuration options."""
    
    @task(
        task_id='custom_extract',
        retries=3,
        retry_delay=timedelta(minutes=5),
        execution_timeout=timedelta(minutes=30),
        pool='default_pool',
        queue='default',
        priority_weight=10,
        do_xcom_push=True,
        multiple_outputs=False,
        executor_config={}
    )
    def extract_with_config():
        """Task with explicit configuration."""
        return {'data': 'extracted'}
    
    @task(
        task_id='custom_transform',
        owner='data_team',
        email=['team@company.com'],
        email_on_failure=True,
        email_on_retry=False
    )
    def transform_with_config(data):
        """Transform with notification settings."""
        return {'transformed': True}
    
    data = extract_with_config()
    transform_with_config(data)

dag_instance = configured_task_dag()

## 3. Automatic XCom Handling

In [ ]:
from airflow.decorators import dag, task
from datetime import datetime

@dag(
    dag_id='xcom_example',
    start_date=datetime(2026, 9, 1),
    schedule_interval='@daily',
    catchup=False
)
def xcom_dag():
    """Demonstrates automatic XCom passing."""
    
    @task
    def produce_data():
        """Return value automatically pushed to XCom."""
        # No need for xcom_push() - return value is automatic
        return {
            'users': 1000,
            'transactions': 5000,
            'revenue': 50000.00
        }
    
    @task
    def consume_data(metrics):
        """Automatically pulls from XCom via parameter."""
        # No need for xcom_pull() - passed as argument
        users = metrics['users']
        revenue = metrics['revenue']
        avg_revenue = revenue / users if users > 0 else 0
        
        print(f'Users: {users}')
        print(f'Revenue: {revenue}')
        print(f'Avg Revenue per User: {avg_revenue:.2f}')
        
        return {'avg_revenue': avg_revenue}
    
    @task
    def generate_report(stats):
        """Final task in pipeline."""
        print(f'Report generated with avg revenue: {stats["avg_revenue"]}')
        return 'report_complete'
    
    # Data flows automatically through XCom
    metrics = produce_data()
    stats = consume_data(metrics)
    generate_report(stats)

dag_instance = xcom_dag()

## 4. Multiple Outputs

In [ ]:
from airflow.decorators import dag, task
from datetime import datetime

@dag(
    dag_id='multiple_outputs_example',
    start_date=datetime(2026, 9, 1),
    schedule_interval='@daily',
    catchup=False
)
def multiple_outputs_dag():
    """Demonstrates returning multiple values from a task."""
    
    @task(multiple_outputs=True)
    def split_data():
        """Return multiple values as separate XComs."""
        # Each key becomes a separate XCom entry
        return {
            'train_data': '/path/to/train.csv',
            'test_data': '/path/to/test.csv',
            'validation_data': '/path/to/val.csv'
        }
    
    @task
    def train_model(train_path):
        """Use only train_data output."""
        print(f'Training with: {train_path}')
        return 'model_trained'
    
    @task
    def evaluate_model(test_path):
        """Use only test_data output."""
        print(f'Evaluating with: {test_path}')
        return 'model_evaluated'
    
    # Access individual outputs by key
    outputs = split_data()
    train_model(outputs['train_data'])
    evaluate_model(outputs['test_data'])

dag_instance = multiple_outputs_dag()

## 5. Task Dependencies and Chaining

In [ ]:
from airflow.decorators import dag, task
from datetime import datetime

@dag(
    dag_id='dependencies_example',
    start_date=datetime(2026, 9, 1),
    schedule_interval='@daily',
    catchup=False
)
def dependencies_dag():
    """Demonstrates different dependency patterns."""
    
    @task
    def extract_a():
        return 'data_a'
    
    @task
    def extract_b():
        return 'data_b'
    
    @task
    def merge(data_a, data_b):
        """Depends on both extracts."""
        return f'{data_a} + {data_b}'
    
    @task
    def transform(merged):
        return f'transformed_{merged}'
    
    @task
    def load_a(transformed):
        return 'loaded_a'
    
    @task
    def load_b(transformed):
        return 'loaded_b'
    
    # Sequential dependencies through function calls
    data_a = extract_a()
    data_b = extract_b()
    merged = merge(data_a, data_b)
    transformed = transform(merged)
    
    # Parallel loads
    load_a(transformed)
    load_b(transformed)

dag_instance = dependencies_dag()

## 6. Using Context and kwargs

In [ ]:
from airflow.decorators import dag, task
from datetime import datetime

@dag(
    dag_id='context_example',
    start_date=datetime(2026, 9, 1),
    schedule_interval='@daily',
    catchup=False
)
def context_dag():
    """Access Airflow context within tasks."""
    
    @task
    def process_with_context(**context):
        """Access execution context."""
        # Available context keys:
        # - execution_date / logical_date
        # - dag_run
        # - task_instance
        # - params
        # - prev_execution_date
        # - next_execution_date
        
        logical_date = context.get('logical_date')
        dag_run = context.get('dag_run')
        params = context.get('params', {})
        
        print(f'Logical date: {logical_date}')
        print(f'DAG run ID: {dag_run.run_id if dag_run else None}')
        print(f'Params: {params}')
        
        return {
            'date': str(logical_date),
            'run_id': dag_run.run_id if dag_run else None
        }
    
    @task
    def use_params(param_value):
        """Use DAG parameters."""
        print(f'Parameter value: {param_value}')
        return param_value
    
    result = process_with_context()
    use_params(result)

dag_instance = context_dag()

## 7. Error Handling and Retries

In [ ]:
from airflow.decorators import dag, task
from datetime import datetime, timedelta
import random

@dag(
    dag_id='error_handling_example',
    start_date=datetime(2026, 9, 1),
    schedule_interval='@daily',
    catchup=False,
    default_args={
        'retries': 3,
        'retry_delay': timedelta(minutes=5),
        'retry_exponential_backoff': True,
    }
)
def error_handling_dag():
    """Demonstrates error handling patterns."""
    
    @task(retries=5, retry_delay=timedelta(seconds=30))
    def flaky_api_call():
        """Simulate unreliable external service."""
        success = random.random() > 0.5
        if not success:
            raise Exception('API call failed - will retry')
        return {'status': 'success'}
    
    @task
    def handle_failure():
        """Fallback task for error scenarios."""
        print('Using fallback data source')
        return {'status': 'fallback'}
    
    @task(trigger_rule='all_done')
    def cleanup():
        """Always runs regardless of upstream status."""
        print('Cleanup completed')
        return 'cleaned'
    
    try:
        data = flaky_api_call()
    except Exception:
        data = handle_failure()
    
    cleanup()

dag_instance = error_handling_dag()

## 8. External Task Operators

In [ ]:
from airflow.decorators import dag, task
from datetime import datetime

@dag(
    dag_id='external_task_example',
    start_date=datetime(2026, 9, 1),
    schedule_interval='@daily',
    catchup=False
)
def external_task_dag():
    """Wait for tasks in other DAGs."""
    
    @task.external_task(
        external_dag_id='upstream_dag',
        external_task_id='final_task',
        execution_delta=timedelta(hours=1),
        check_existence=True
    )
    def wait_for_upstream():
        """Waits for external task completion."""
        pass
    
    @task
    def process_after_wait():
        """Runs after upstream completes."""
        print('Upstream complete, processing now')
        return 'processed'
    
    wait_for_upstream()
    process_after_wait()

dag_instance = external_task_dag()

## 9. Task Groups with @task

In [ ]:
from airflow.decorators import dag, task
from airflow.utils.task_group import TaskGroup
from datetime import datetime

@dag(
    dag_id='task_group_example',
    start_date=datetime(2026, 9, 1),
    schedule_interval='@daily',
    catchup=False
)
def task_group_dag():
    """Organize tasks into logical groups."""
    
    @task
    def start():
        return 'started'
    
    with TaskGroup('processing_group') as processing:
        @task
        def extract():
            return 'extracted'
        
        @task
        def transform(data):
            return f'transformed_{data}'
        
        @task
        def validate(data):
            return f'validated_{data}'
        
        ext = extract()
        trans = transform(ext)
        validate(trans)
    
    @task
    def finish():
        return 'finished'
    
    start() >> processing >> finish()

dag_instance = task_group_dag()

## 10. Best Practices and Common Patterns

In [ ]:
from airflow.decorators import dag, task
from datetime import datetime, timedelta
import logging

logger = logging.getLogger(__name__)

@dag(
    dag_id='best_practices_task_dag',
    start_date=datetime(2026, 9, 1),
    schedule_interval='@daily',
    catchup=False,
    default_args={
        'owner': 'data_team',
        'retries': 2,
        'retry_delay': timedelta(minutes=5),
    }
)
def best_practices_dag():
    """Demonstrates @task best practices."""
    
    # PRACTICE 1: Keep tasks focused and small
    @task(task_id='extract_clean')
    def extract_data():
        """Single responsibility: extract only."""
        logger.info('Starting extraction')
        # Extract logic here
        return {'rows': 1000}
    
    # PRACTICE 2: Use type hints for clarity
    @task
    def transform_data(extracted: dict) -> dict:
        """Type hints improve readability."""
        rows = extracted.get('rows', 0)
        logger.info(f'Transforming {rows} rows')
        return {'transformed_rows': rows}
    
    # PRACTICE 3: Handle errors gracefully
    @task
    def load_data(transformed: dict) -> str:
        """Include error handling."""
        try:
            rows = transformed.get('transformed_rows', 0)
            logger.info(f'Loading {rows} rows')
            # Load logic here
            return 'success'
        except Exception as e:
            logger.error(f'Load failed: {e}')
            raise
    
    # PRACTICE 4: Use meaningful task IDs
    @task(task_id='generate_summary_report')
    def report(status: str) -> None:
        """Descriptive task ID explains purpose."""
        logger.info(f'Report generated: {status}')
    
    # PRACTICE 5: Chain tasks clearly
    extracted = extract_data()
    transformed = transform_data(extracted)
    status = load_data(transformed)
    report(status)

dag_instance = best_practices_dag()

## Summary

### @task Parameters

| Parameter | Purpose |
|-----------|---------|
| `task_id` | Custom task identifier |
| `retries` | Number of retry attempts |
| `retry_delay` | Time between retries |
| `execution_timeout` | Max execution time |
| `pool` | Resource pool assignment |
| `queue` | Queue for task execution |
| `priority_weight` | Execution priority |
| `do_xcom_push` | Enable XCom pushing |
| `multiple_outputs` | Return dict as separate XComs |
| `owner` | Task owner |
| `email` | Notification emails |

### Best Practices

✓ Keep tasks small and focused  
✓ Use type hints for clarity  
✓ Add meaningful task IDs  
✓ Handle errors with try/except  
✓ Use logging instead of print  
✓ Leverage automatic XCom handling  
✓ Set appropriate timeouts and retries  
✓ Document task purpose in docstrings